In [ ]:
!pip install spikingjelly brian2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

# Train LIF

In [1]:
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
from torch.cuda import amp
import torchvision
import numpy as np

from spikingjelly.activation_based import neuron, encoding, functional, surrogate, layer


class SNN(nn.Module):
    def __init__(self, tau):
        super().__init__()

        self.layer = nn.Sequential(
            layer.Flatten(),
            layer.Linear(8 * 8, 10, bias=False),
            # neuron.LIFNode(tau=tau, surrogate_function=surrogate.ATan()),
            neuron.IzhikevichNode(tau=tau, surrogate_function=surrogate.ATan()),
            )

    def forward(self, x: torch.Tensor):
        return self.layer(x)

In [2]:
args = {
    'T': 100,                         # simulating time-steps
    'device': 'cpu',               # device
    'b': 64,                          # batch size
    'epochs': 10,                      # number of total epochs to run
    'j': 4,                           # number of data loading workers
    'data_dir': './mnist',            # root dir of MNIST dataset (customize as needed)
    'out_dir': './logs',              # root dir for saving logs and checkpoint
    'resume': None,                   # resume from the checkpoint path
    'amp': False,                     # automatic mixed precision training
    'opt': 'adam',                    # optimizer: 'sgd' or 'adam'
    'momentum': 0.9,                  # momentum for SGD
    'lr': 1e-3,                       # learning rate
    'tau': 2.0                        # parameter tau of LIF neuron
}

# Access like: config['epochs'], config['lr'], etc.

net = SNN(tau=args['tau'])

print(net)

net.to(args['device'])

# 初始化数据加载器
transform = torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Resize((8, 8))
    ])

train_dataset = torchvision.datasets.MNIST(
    root=args['data_dir'],
    train=True,
    transform=transform,
    download=True
)
test_dataset = torchvision.datasets.MNIST(
    root=args['data_dir'],
    train=False,
    transform=transform,
    download=True
)

train_data_loader = data.DataLoader(
    dataset=train_dataset,
    batch_size=args['b'],
    shuffle=True,
    drop_last=True,
    num_workers=args['j'],
    pin_memory=True
)
test_data_loader = data.DataLoader(
    dataset=test_dataset,
    batch_size=args['b'],
    shuffle=False,
    drop_last=False,
    num_workers=args['j'],
    pin_memory=True
)

scaler = None
if args['amp']:
    scaler = amp.GradScaler()

start_epoch = 0
max_test_acc = -1

optimizer = None
if args['opt'] == 'sgd':
    optimizer = torch.optim.SGD(net.parameters(), lr=args['lr'], momentum=args['momentum'])
elif args['opt'] == 'adam':
    optimizer = torch.optim.Adam(net.parameters(), lr=args['lr'])
else:
    raise NotImplementedError(args['opt'])

if args['resume']:
    checkpoint = torch.load(args['resume'], map_location='cpu')
    net.load_state_dict(checkpoint['net'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    start_epoch = checkpoint['epoch'] + 1
    max_test_acc = checkpoint['max_test_acc']

out_dir = "/content/snn-lif"

if args['amp']:
    out_dir += '_amp'

if not os.path.exists(out_dir):
    os.makedirs(out_dir)
    print(f'Mkdir {out_dir}.')

# with open(os.path.join(out_dir, 'args['txt']'), 'w', encoding='utf-8') as args_txt:
#     args_txt.write(str(args))

# writer = SummaryWriter(out_dir, purge_step=start_epoch)
# with open(os.path.join(out_dir, 'args['txt']'), 'w', encoding='utf-8') as args_txt:
#     args_txt.write(str(args))
#     args_txt.write('\n')
#     args_txt.write(' '.join(sys.argv))

encoder = encoding.PoissonEncoder()

for epoch in range(start_epoch, args['epochs']):
    start_time = time.time()
    net.train()
    train_loss = 0
    train_acc = 0
    train_samples = 0
    for img, label in train_data_loader:
        optimizer.zero_grad()
        img = img.to(args['device'])
        label = label.to(args['device'])
        label_onehot = F.one_hot(label, 10).float()

        if scaler is not None:
            with amp.autocast():
                out_fr = 0.
                for t in range(args['T']):
                    encoded_img = encoder(img)
                    out_fr += net(encoded_img)
                out_fr = out_fr / args['T']
                loss = F.mse_loss(out_fr, label_onehot)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            out_fr = 0.
            for t in range(args['T']):
                encoded_img = encoder(img)
                out_fr += net(encoded_img)
            out_fr = out_fr / args['T']
            loss = F.mse_loss(out_fr, label_onehot)
            loss.backward()
            optimizer.step()

        train_samples += label.numel()
        train_loss += loss.item() * label.numel()
        train_acc += (out_fr.argmax(1) == label).float().sum().item()

        functional.reset_net(net)

    train_time = time.time()
    train_speed = train_samples / (train_time - start_time)
    train_loss /= train_samples
    train_acc /= train_samples

    # writer.add_scalar('train_loss', train_loss, epoch)
    # writer.add_scalar('train_acc', train_acc, epoch)

    net.eval()
    test_loss = 0
    test_acc = 0
    test_samples = 0
    with torch.no_grad():
        for img, label in test_data_loader:
            img = img.to(args['device'])
            label = label.to(args['device'])
            label_onehot = F.one_hot(label, 10).float()
            out_fr = 0.
            for t in range(args['T']):
                encoded_img = encoder(img)
                out_fr += net(encoded_img)
            out_fr = out_fr / args['T']
            loss = F.mse_loss(out_fr, label_onehot)

            test_samples += label.numel()
            test_loss += loss.item() * label.numel()
            test_acc += (out_fr.argmax(1) == label).float().sum().item()
            functional.reset_net(net)
    test_time = time.time()
    test_speed = test_samples / (test_time - train_time)
    test_loss /= test_samples
    test_acc /= test_samples
    # writer.add_scalar('test_loss', test_loss, epoch)
    # writer.add_scalar('test_acc', test_acc, epoch)

    save_max = False
    if test_acc > max_test_acc:
        max_test_acc = test_acc
        save_max = True

    checkpoint = {
        'net': net.state_dict(),
        'optimizer': optimizer.state_dict(),
        'epoch': epoch,
        'max_test_acc': max_test_acc
    }

    if save_max:
        torch.save(checkpoint, os.path.join(out_dir, 'checkpoint_max.pth'))

    torch.save(checkpoint, os.path.join(out_dir, 'checkpoint_latest.pth'))

    print(args)
    print(out_dir)
    print(f'epoch ={epoch}, train_loss ={train_loss: .4f}, train_acc ={train_acc: .4f}, test_loss ={test_loss: .4f}, test_acc ={test_acc: .4f}, max_test_acc ={max_test_acc: .4f}')
    print(f'train speed ={train_speed: .4f} images/s, test speed ={test_speed: .4f} images/s')
    # print(f'escape time = {(datetime.datetime.now() + datetime.timedelta(seconds=(time.time() - start_time) * (args['epochs'] - epoch))).strftime("%Y-%m-%d %H:%M:%S")}\n')

# 保存绘图用数据
net.eval()
# 注册钩子
output_layer = net.layer[-1] # 输出层
output_layer.v_seq = []
output_layer.s_seq = []
def save_hook(m, x, y):
    m.v_seq.append(m.v.unsqueeze(0))
    m.s_seq.append(y.unsqueeze(0))

output_layer.register_forward_hook(save_hook)


with torch.no_grad():
    img, label = test_dataset[0]
    img = img.to(args['device'])
    out_fr = 0.
    for t in range(args['T']):
        encoded_img = encoder(img)
        out_fr += net(encoded_img)
    out_spikes_counter_frequency = (out_fr / args['T']).cpu().numpy()
    print(f'Firing rate: {out_spikes_counter_frequency}')

    output_layer.v_seq = torch.cat(output_layer.v_seq)
    output_layer.s_seq = torch.cat(output_layer.s_seq)
    v_t_array = output_layer.v_seq.cpu().numpy().squeeze()  # v_t_array[i][j]表示神经元i在j时刻的电压值
    np.save("v_t_array.npy",v_t_array)
    s_t_array = output_layer.s_seq.cpu().numpy().squeeze()  # s_t_array[i][j]表示神经元i在j时刻释放的脉冲，为0或1
    np.save("s_t_array.npy",s_t_array)

SNN(
  (layer): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1, step_mode=s)
    (1): Linear(in_features=64, out_features=10, bias=False)
    (2): IzhikevichNode(
      v_threshold=1.0, v_reset=0.0, detach_reset=False, step_mode=s, backend=torch, v_rest=-0.1, w_rest=0.0, tau_w=2.0, a=0.0, b=0.0, tau=2.0, v_c=0.8, a0=1.0
      (surrogate_function): ATan(alpha=2.0, spiking=True)
    )
  )
)
{'T': 100, 'device': 'cpu', 'b': 64, 'epochs': 10, 'j': 4, 'data_dir': './mnist', 'out_dir': './logs', 'resume': None, 'amp': False, 'opt': 'adam', 'momentum': 0.9, 'lr': 0.001, 'tau': 2.0}
/content/snn-lif
epoch =0, train_loss = 0.0655, train_acc = 0.7042, test_loss = 0.0528, test_acc = 0.8106, max_test_acc = 0.8106
train speed = 852.3867 images/s, test speed = 933.5383 images/s
{'T': 100, 'device': 'cpu', 'b': 64, 'epochs': 10, 'j': 4, 'data_dir': './mnist', 'out_dir': './logs', 'resume': None, 'amp': False, 'opt': 'adam', 'momentum': 0.9, 'lr': 0.001, 'tau': 2.0}
/content/snn-lif
epoch =1, tr

KeyboardInterrupt: 

In [4]:
net

SNN(
  (layer): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1, step_mode=s)
    (1): Linear(in_features=64, out_features=64, bias=False)
    (2): Linear(in_features=64, out_features=10, bias=False)
    (3): LIFNode(
      v_threshold=1.0, v_reset=0.0, detach_reset=False, step_mode=s, backend=torch, tau=2.0
      (surrogate_function): ATan(alpha=2.0, spiking=True)
    )
  )
)

In [85]:
layer_weights = []
net.eval() 
for i in range(len(net.cpu().layer)):
    if (net.cpu().layer[i]._get_name() == 'Linear'):
        layer_weights.append(net.cpu().layer[i].weight.detach().numpy())
np.savez('outputs/weights-1layer.npz', *layer_weights)

loaded = np.load('outputs/weights-1layer.npz')
loaded_list = [loaded[key] for key in loaded.files]

# HH from LIF

In [86]:
from brian2 import *
import numpy as np

loaded = np.load('outputs/weights-1layer.npz')
layer_weights = [loaded[key] for key in loaded.files]

@check_units(v=volt, result=Hz)
def alpha_m(v):
    dv = (25*mV - v)
    return clip(dv/(exp(dv/(10*mV)) - 1), -1e6, 1e6) * (0.1/mV) / ms

C = 1*uF/cm**2
gNa = 120*msiemens/cm**2
gK = 36*msiemens/cm**2
gL = 0.3*msiemens/cm**2
ENa = 120*mV
EK = -12*mV
EL = -10.5*mV

hh_eqs = '''
dv/dt = (I_inj - INa - IK - IL) / C : volt
INa = gNa * m**3 * h * (v - ENa) : amp/meter**2
IK = gK * n**4 * (v - EK) : amp/meter**2
IL = gL * (v - EL) : amp/meter**2

dm/dt = alpham*(1-m) - betam*m : 1
dn/dt = alphan*(1-n) - betan*n : 1
dh/dt = alphah*(1-h) - betah*h : 1

alpham = 0.1/mV*(25*mV - v)/(exp((25*mV - v)/(10*mV)) - 1)/ms : Hz (constant over dt)
betam = 4*exp(-v/(18*mV))/ms : Hz (constant over dt)
alphah = 0.07*exp(-v/(20*mV))/ms : Hz (constant over dt)
betah = 1/(exp((30*mV - v)/(10*mV)) + 1)/ms : Hz (constant over dt)
alphan = 0.01/mV*(10*mV - v)/(exp((10*mV - v)/(10*mV)) - 1)/ms : Hz (constant over dt)
betan = 0.125*exp(-v/(80*mV))/ms : Hz (constant over dt)
I_inj : amp/meter**2
'''


device.reinit()
device.activate()
defaultclock.dt = 0.1*ms

In [ ]:
train_dataset[1]

(tensor([[[0.0000e+00, 0.0000e+00, 0.0000e+00, 2.8476e-04, 2.6020e-02,
           2.5947e-02, 2.7918e-04, 0.0000e+00],
          [0.0000e+00, 0.0000e+00, 1.0437e-03, 8.9156e-02, 5.3961e-01,
           4.8133e-01, 4.0803e-02, 0.0000e+00],
          [0.0000e+00, 1.8620e-04, 8.7210e-02, 5.9194e-01, 7.3025e-01,
           6.0672e-01, 2.3265e-01, 0.0000e+00],
          [0.0000e+00, 2.6788e-02, 4.4737e-01, 4.0749e-01, 1.0228e-01,
           3.1810e-01, 4.3972e-01, 0.0000e+00],
          [0.0000e+00, 1.4584e-01, 5.1816e-01, 3.2705e-02, 1.7641e-02,
           3.7345e-01, 3.4693e-01, 0.0000e+00],
          [0.0000e+00, 1.7828e-01, 5.1886e-01, 1.9326e-01, 3.9992e-01,
           3.4986e-01, 4.1984e-02, 0.0000e+00],
          [0.0000e+00, 9.2114e-02, 5.8795e-01, 5.9092e-01, 2.7039e-01,
           3.2743e-02, 0.0000e+00, 0.0000e+00],
          [0.0000e+00, 1.4127e-03, 4.0135e-02, 4.1682e-02, 1.8203e-03,
           0.0000e+00, 0.0000e+00, 0.0000e+00]]]),
 0)

In [55]:
import torchvision

transform = torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Resize((8, 8))
    ])

test_dataset = torchvision.datasets.MNIST(
    root="./mnist",
    train=False,
    transform=transform,
    download=True
)


In [ ]:
import os
from torchvision.utils import save_image

def save_test_dataset(test_dataset, save_dir="saved_dataset"):
    os.makedirs(save_dir, exist_ok=True)

    for idx in range(100):
        img, label = test_dataset[idx]  
        img_path = os.path.join(save_dir, f"img_{idx}_{label}.png")
        save_image(img, img_path)
save_test_dataset(test_dataset, save_dir="saved_dataset")

In [43]:
layer_weights[0].shape[0]

10

In [ ]:
import random

def run(index):
    img, label = test_dataset[index]
    img = img.unsqueeze(0).to('cpu')

    T = 100
    encoder = encoding.PoissonEncoder()
    spike_trains = []
    for t in range(T):
        encoded_img = encoder(img)
        spike_trains.append(encoded_img.view(-1).squeeze().numpy())

    spike_indices = []
    spike_times = []
    for t in range(T):
        dt = defaultclock.dt
        time_in_ms = t * dt
        for neuron_idx, spike in enumerate(spike_trains[t]):
            if spike == 1:
                spike_indices.append(neuron_idx)
                spike_times.append(time_in_ms)
    spike_indices = np.array(spike_indices)
    spike_times = np.array(spike_times) * second
    input_size = layer_weights[0].shape[1]
    input_layer = SpikeGeneratorGroup(input_size, spike_indices, spike_times)

    threshold = 'v > -55*mV'
    reset = ''
    refractory = 2*ms
    scale = 1 * amp / cm**2

    layers = [input_layer]
    neuron_groups = []
    synapses = []

    for i, w in enumerate(layer_weights):
        out_size, in_size = w.shape

        # print(in_size, out_size)

        group = NeuronGroup(out_size, hh_eqs,
                            threshold=threshold,
                            reset=reset,
                            refractory=refractory,
                            method='exponential_euler')

        group.v = -65*mV
        group.m = 0.05
        group.h = 0.6
        group.n = 0.32
        group.I_inj = 0*amp/cm**2

        if i == 0:
            neuron_idx = random.randint(0, out_size - 1)
            mon_voltage = StateMonitor(group, 'v', record=[neuron_idx])

        neuron_groups.append(group)
        layers.append(group)

        syn = Synapses(layers[i], group, model='w : amp/meter**2', on_pre='I_inj_post += w')
        syn.connect(True)
        syn.w = w.T.flatten() * scale
        synapses.append(syn)

    mon_output = SpikeMonitor(layers[-1])

    all_components = [input_layer] + neuron_groups + synapses + [mon_output]
    net = Network(*all_components)
    net.run(T * defaultclock.dt)

    final_layer_size = layer_weights[-1].shape[0]
    spike_counts = np.array([mon_output.count[i] for i in range(final_layer_size)])
    predicted = np.argmax(spike_counts)

    # import matplotlib.pyplot as plt
    # plt.figure(figsize=(8, 4))
    # plt.plot(mon_voltage.t / ms, mon_voltage.v[0] / mV)
    # plt.xlabel('Time (ms)')
    # plt.ylabel('Membrane potential (mV)')
    # plt.title('Voltage trace of monitored neuron')
    # plt.show()
    
    return predicted, label

In [88]:
correct = 0
samples = 100
np.seterr(divide='ignore', invalid='ignore', over='ignore')

for i in range(samples):
    pred, label = run(i)
    print(f"Sample {i}: True = {label}, Pred = {pred}")
    if pred == label:
        correct += 1

print(f"Accuracy: {correct/samples:.2%}")

Sample 0: True = 7, Pred = 3
Sample 1: True = 2, Pred = 2
Sample 2: True = 1, Pred = 1
Sample 3: True = 0, Pred = 0
Sample 4: True = 4, Pred = 4
Sample 5: True = 1, Pred = 1
Sample 6: True = 4, Pred = 4
Sample 7: True = 9, Pred = 4
Sample 8: True = 5, Pred = 4
Sample 9: True = 9, Pred = 8
Sample 10: True = 0, Pred = 0
Sample 11: True = 6, Pred = 3
Sample 12: True = 9, Pred = 4
Sample 13: True = 0, Pred = 0
Sample 14: True = 1, Pred = 1
Sample 15: True = 5, Pred = 3
Sample 16: True = 9, Pred = 2
Sample 17: True = 7, Pred = 7
Sample 18: True = 3, Pred = 3
Sample 19: True = 4, Pred = 4
Sample 20: True = 9, Pred = 9
Sample 21: True = 6, Pred = 6
Sample 22: True = 6, Pred = 0
Sample 23: True = 5, Pred = 5
Sample 24: True = 4, Pred = 9
Sample 25: True = 0, Pred = 0
Sample 26: True = 7, Pred = 7
Sample 27: True = 4, Pred = 4
Sample 28: True = 0, Pred = 0
Sample 29: True = 1, Pred = 1
Sample 30: True = 3, Pred = 3
Sample 31: True = 1, Pred = 1
Sample 32: True = 3, Pred = 3
Sample 33: True = 4,

# save ds

In [ ]:
def get_image(index):
    img, label = test_dataset[index]
    img = img.unsqueeze(0)
    img = img.to('cpu')

    from spikingjelly.activation_based import encoding
    encoder = encoding.PoissonEncoder()
    T = 100
    spike_trains = []
    for t in range(T):
        encoded_img = encoder(img)
        spike_trains.append(encoded_img.view(-1).squeeze().numpy())
    return spike_trains, label

spike_imgs = {}
for i in range(10):
  spike_imgs[i] = []
images = []
for i in range(100):
  spike_trains, label=get_image(i)
  spike_list=[]
  for frame in spike_trains:
    spike_list.append(frame.astype(int).tolist())
  spike_imgs[label].append(spike_list)

In [63]:
from PIL import Image
import os
import numpy as np
from torch.utils.data import DataLoader

test_loader = DataLoader(test_dataset, batch_size=64)
output_dir = "test_images_png"
os.makedirs(output_dir, exist_ok=True)

for idx, (spike_trains, label) in enumerate(test_loader):
    for b in range(spike_trains.size(0)):  # handle batch size
        frames = spike_trains[b]  # (time, H, W) or similar
        for t, frame in enumerate(frames):
            img_array = np.array(frame * 255, dtype=np.uint8)  # scale to 0–255
            img = Image.fromarray(img_array)
            filename = f"test_{idx}_{b}_{t}_{label[b].item()}.png"
            img.save(os.path.join(output_dir, filename))

In [ ]:
output_dir = "./outputs/output_images"
import os
os.makedirs(output_dir, exist_ok=True)

for label, image_list in spike_imgs.items():
    for idx, image_data in enumerate(image_list):
        filename = f"img-{label}-{idx}.txt"
        filepath = os.path.join(output_dir, filename)
        with open(filepath, 'w') as f:
            f.write(str(image_data))

In [ ]:
import shutil
shutil.make_archive("./outputs/output_images", 'zip', "./outputs/output_images")

'c:\\Users\\arian\\Documents\\hh\\HH\\Train\\output_images.zip'